<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [ ]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [ ]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [ ]:
import numpy as np
import pandas as pd

# Basic structure
print("Shape (rows, cols):", df.shape)
display(df.head())

# Column types + non-null counts
df.info()

# Missing values summary
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)

summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing_count": missing_count,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

display(summary)

<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [ ]:
num_stats = df.describe(include=[np.number]).T
display(num_stats)

# Optional: quick check for extreme values
# display(df.select_dtypes(include=[np.number]).quantile([0.01, 0.5, 0.99]).T)

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [ ]:
# Look at missing and odd Country entries
print("Missing Country:", df["Country"].isna().sum())

# Show most frequent countries
display(df["Country"].value_counts(dropna=False).head(25))

# Detect potential inconsistencies (leading/trailing spaces, weird casing)
country_raw = df["Country"].dropna().astype(str)
has_leading_trailing_space = country_raw[country_raw.str.startswith(" ") | country_raw.str.endswith(" ")]

print("Countries with leading/trailing spaces (sample):")
display(has_leading_trailing_space.unique()[:20])


<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [ ]:
# --- Standardize Country ---
df["Country"] = df["Country"].astype("string")

# Strip whitespace
df["Country"] = df["Country"].str.strip()

# Map common variants (adjust/add if your dataset contains these)
country_map = {
    "United States": "United States of America",
    "USA": "United States of America",
    "U.S.A.": "United States of America",
    "US": "United States of America",
    "U.K.": "United Kingdom",
    "UK": "United Kingdom",
    "Russian Federation": "Russia",
}

df["Country"] = df["Country"].replace(country_map)

# --- Standardize EdLevel (optional cleanup) ---
if "EdLevel" in df.columns:
    df["EdLevel"] = df["EdLevel"].astype("string").str.strip()

print("Cleaned Country examples:")
display(df["Country"].value_counts().head(20))

### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [ ]:
# One-hot encode Employment (keep NaN as its own category so you don't lose info)
employment_dummies = pd.get_dummies(df["Employment"], prefix="Employment", dummy_na=True)

# Add encoded columns to df (or keep separate)
df_encoded = pd.concat([df, employment_dummies], axis=1)

print("Original df shape:", df.shape)
print("Encoded df shape:", df_encoded.shape)
display(df_encoded.head())

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
})

display(missing_table.head(20))

<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [ ]:
num_col = "ConvertedCompYearly"

if num_col in df.columns:
    before = df[num_col].isna().sum()
    median_val = df[num_col].median(skipna=True)
    
    df[num_col] = df[num_col].fillna(median_val)
    
    after = df[num_col].isna().sum()
    print(f"{num_col}: missing before={before}, after={after}, imputed with median={median_val}")
else:
    print(f"Column '{num_col}' not found.")

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [ ]:
cat_col = "RemoteWork"

if cat_col in df.columns:
    before = df[cat_col].isna().sum()
    mode_val = df[cat_col].mode(dropna=True)
    mode_val = mode_val.iloc[0] if len(mode_val) > 0 else None

    if mode_val is not None:
        df[cat_col] = df[cat_col].fillna(mode_val)

    after = df[cat_col].isna().sum()
    print(f"{cat_col}: missing before={before}, after={after}, filled with mode='{mode_val}'")
else:
    print(f"Column '{cat_col}' not found.")

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [ ]:
col = "ConvertedCompYearly"

if col in df.columns:
    
    min_val = df[col].min()
    max_val = df[col].max()
    
    df[col + "_MinMax"] = (df[col] - min_val) / (max_val - min_val)

    display(df[[col, col + "_MinMax"]].head(10))
    
else:
    print("Column not found")

<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [ ]:
col = "ConvertedCompYearly"

if col in df.columns:
    # log1p handles zeros safely (log(1+x))
    df[col + "_Log"] = np.log1p(df[col])
    display(df[[col, col + "_Log"]].head(10))
else:
    print(f"Column '{col}' not found.")

### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [ ]:
import numpy as np
import pandas as pd

col = "YearsCodePro"

if col in df.columns:
    y = df[col].astype("string").str.strip()

    # Convert text categories to numbers
    y = y.replace({
        "Less than 1 year": "0",
        "More than 50 years": "51"
    })

    years = pd.to_numeric(y, errors="coerce")  # NaN if not convertible

    # Define bins (you can adjust thresholds if your lab suggests different ones)
    def to_level(v):
        if pd.isna(v):
            return np.nan
        if v <= 2:
            return "Junior"
        if 3 <= v <= 7:
            return "Mid"
        if 8 <= v <= 15:
            return "Senior"
        return "Expert"

    df["ExperienceLevel"] = years.apply(to_level)

    display(df[[col, "ExperienceLevel"]].head(15))
    print(df["ExperienceLevel"].value_counts(dropna=False))
else:
    print(f"Column '{col}' not found.")

### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
